# 02 · Signal exploration

## Purpose
Look at the signals before any feature exists, and set the few physics parameters Stage 2 needs:
the microphone burst threshold (and prove bursts are an artefact worth removing), the envelope demodulation band per
radial accelerometer, and the bearing comb definitions. It also checks the one recording whose speed estimates
disagree, and audits whether `cage_fault` really looks like a cage fault.

Rule 6 applies throughout: only **train** recordings (`split_closed == "train"`) set a parameter. Train + val are used
for descriptive checks. Test recordings are never loaded.

## Inputs
`metadata.csv`, `data/raw_npy/`, `configs/base.yaml`, and the caches written by `scripts/03_signal_scan.py`
(named by the sha256 of the config sections that produced them).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the tools this notebook needs, reads <code>metadata.csv</code> and the settings, and finds the cached results that <code>scripts/03_signal_scan.py</code> computed earlier. If a cache is missing, it stops and says which scan to run. It also defines two small helpers: one loads a recording, one measures how far each sample is from normal (the robust z-score).</div>

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score

from ml.config import config_sha256, load_config
from ml.features import amplitude_spectrum, comb_lines, line_prominence_db, order_spectrum, select_demod_band
from ml.labels import CLASS10
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR, cache_file, data_dir
from ml.preprocess import remove_mic_bursts, speed_candidates, tacho_intervals

cfg = load_config(DEFAULT_CONFIG)
pre = cfg.preprocess
FS = pre.fs_hz
CH = list(pre.channels)
MIC = CH.index("mic")
ACCELS = [CH.index(c) for c in CH if c not in ("tacho", "mic")]
FIG = SYSTEM_CODE_DIR / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

meta = pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")
train = meta[meta.split_closed == "train"]
trval = meta[meta.split_closed != "test"]

caches = {
    "mic": cache_file("02_mic", pre.mic, ext="parquet"),
    "meanspec": cache_file("02_meanspec", pre.mic, ext="npz"),
    "sk": cache_file("02_sk", pre.envelope.sk, ext="parquet"),
    "envelope": cache_file("02_envelope", pre.envelope, ext="npz"),
    "sensor": cache_file("02_sensor", ext="parquet"),
    "bandcompare": cache_file("02_bandcompare", pre.envelope, ext="npz"),
}
missing = [k for k, p in caches.items() if not p.exists()]
assert not missing, f"run: python scripts/03_signal_scan.py {' '.join(missing)}"


def load(rec_id: str) -> np.ndarray:
    return np.load(data_dir() / "raw_npy" / f"{rec_id}.npy")


def robust_z(x: np.ndarray) -> np.ndarray:
    med = np.median(x)
    return np.abs(x - med) / (1.4826 * np.median(np.abs(x - med)))


print(f"preprocess config sha256 {config_sha256(pre)[:16]}…")
for k, p in caches.items():
    print(f"cache {k:9s} {p.name}")

## (a) Microphone bursts

### Where the bursts are
`peakz_mic` is the largest robust z-score |x − median| / (1.4826 · MAD) of the whole microphone recording, computed at
indexing. A burst-free microphone stays in single digits. Log scale, train + val, one panel per class, so sub-threshold
bursts in other classes would show as a tail.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Draws one histogram per class of each recording's largest microphone spike score, on a log scale. Recordings with bursts sit far to the right; the red dashed line is the burst threshold.</div>

In [ ]:
bins = np.logspace(np.log10(2), np.log10(3000), 60)
fig, axes = plt.subplots(2, 5, figsize=(15, 5.5), sharex=True)
for ax, cls in zip(axes.flat, CLASS10):
    v = trval.loc[trval.class10 == cls, "peakz_mic"]
    ax.hist(v, bins=bins, color="#4c72b0")
    ax.axvline(pre.mic.burst_z, color="red", lw=1, ls="--")
    ax.set(xscale="log", title=f"{cls} (n={len(v)})")
for ax in axes[-1]:
    ax.set_xlabel("peak robust z of microphone")
for ax in axes[:, 0]:
    ax.set_ylabel("recordings")
fig.suptitle(f"Microphone peak robust z, train + val (red: burst threshold {pre.mic.burst_z})")
fig.tight_layout()
fig.savefig(FIG / "02_mic_peakz_hist.png")

### The threshold
Rule: **z_thresh = 1.25 × the largest `peakz_mic` among `split_closed` TRAIN normal recordings**, rounded to 0.1.
Train normal recordings define "no bursts"; the 25 % margin keeps ordinary loud moments out. The check below confirms the
value falls in the gap between burst-free and bursty files, and lists exactly which files it catches (all splits,
from the indexed statistic; no test signal is loaded).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Works out the burst threshold from the rule (1.25 × the largest spike score among training normal recordings) and checks it equals the value stored in the settings. It then counts which files the threshold catches, per class and severity, and saves the full list to <code>results/mic_burst_files.csv</code>.</div>

In [ ]:
tn = train.loc[train.class10 == "normal", "peakz_mic"]
rule_z = round(1.25 * tn.max(), 1)
print(f"train normal: n={len(tn)}, max {tn.max():.2f}, p95 {tn.quantile(.95):.2f} → rule z_thresh {rule_z}")
assert rule_z == pre.mic.burst_z, "configs/base.yaml mic.burst_z is out of date"

below = meta.loc[meta.peakz_mic <= pre.mic.burst_z, "peakz_mic"]
caught = meta[meta.peakz_mic > pre.mic.burst_z].sort_values(["class10", "severity", "fr_file_hz"])
print(f"gap: largest peakz below threshold {below.max():.2f}, smallest above {caught.peakz_mic.min():.2f}")
print(f"caught: {len(caught)} files ({(caught.split_closed != 'test').sum()} train+val)")
caught[["rec_id", "class10", "severity", "split_closed", "peakz_mic"]].to_csv(
    SYSTEM_CODE_DIR / "results" / "mic_burst_files.csv", index=False)
pd.crosstab([caught.class10, caught.severity], caught.split_closed, margins=True)

The weakest caught files, and the loudest burst-free ones on either side of the gap:

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows the three loudest recordings that stay under the threshold next to the three quietest ones above it, so you can see the gap the threshold sits in.</div>

In [ ]:
pd.concat([meta.loc[below.sort_values().index[-3:], ["rec_id", "class10", "split_closed", "peakz_mic"]],
           caught.nsmallest(3, "peakz_mic")[["rec_id", "class10", "split_closed", "peakz_mic"]]])

### What a burst looks like
Three train bursts at sample level: the largest burst in one imbalance file, one underhang cage file and the lone overhang
cage file, ±2 ms around it. An electrical glitch lasts a few samples, has no ring-down, and has no matching accelerometer
response; an acoustic event would ring and would usually shake the frame too.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Zooms in on one burst in each of three recordings, 2 ms either side. The top row is the microphone; the bottom row is the six accelerometers. If the bursts were real sounds from the machine, the accelerometers would react too.</div>

In [ ]:
burst_train = trval[(trval.peakz_mic > pre.mic.burst_z)]
examples = [burst_train[burst_train.class10 == "imbalance"].sort_values("peakz_mic").iloc[len(burst_train) // 4].rec_id,
            burst_train[burst_train.class10 == "underhang_cage"].sort_values("peakz_mic").iloc[20].rec_id]
oc = burst_train[burst_train.class10 == "overhang_cage"]
examples.append(oc.rec_id.iloc[0] if len(oc) else burst_train.sort_values("peakz_mic").rec_id.iloc[-1])

half = int(0.002 * FS)
fig, axes = plt.subplots(2, 3, figsize=(15, 5.5), sharex=True)
for j, rid in enumerate(examples):
    x = load(rid)
    k = int(np.argmax(np.abs(x[:, MIC] - np.median(x[:, MIC]))))
    sl = slice(max(k - half, 0), k + half)
    t = (np.arange(x.shape[0])[sl] - k) / FS * 1e3
    axes[0, j].plot(t, x[sl, MIC], lw=0.8, color="#8172b2")
    axes[0, j].set(title=rid, ylabel="microphone (V)")
    for c in ACCELS:
        axes[1, j].plot(t, robust_z(x[:, c])[sl], lw=0.6, label=CH[c])
    axes[1, j].axhline(pre.mic.burst_z, color="red", ls="--", lw=0.8)
    axes[1, j].set(xlabel="time from burst peak (ms)", ylabel="accelerometer robust z")
axes[1, 0].legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig(FIG / "02_mic_burst_zoom.png")

The same questions over every train + val burst file (from the scan cache): run length in samples, the largest
accelerometer robust z within ±2 ms of each burst against the same statistic at 50 random times, and the microphone RMS
just after the burst relative to the whole recording (ring-down ≫ 1 would mean an acoustic event).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Summarises every burst recording from the cache: how many bursts, how long they last (in samples), how much the accelerometers react at a burst compared with random moments, and whether the microphone keeps ringing afterwards.</div>

In [ ]:
mic = pd.read_parquet(caches["mic"])
b = mic[mic.n_bursts > 0]
cols = ["n_bursts", "run_len_median", "run_len_max", "accel_z_at_burst", "accel_z_random", "ringdown_ratio"]
b[cols].describe(percentiles=[.5]).T[["count", "min", "50%", "max"]].round(2)

### Removing them
`ml.preprocess.remove_mic_bursts` flags samples above the threshold, widens each run by ±5 samples and interpolates
linearly across it. One of the bursty train files, before and after:

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Runs the burst-removal function on one recording and plots the microphone before (grey) and after (purple): all 5 seconds on the left, a zoom on the largest burst on the right.</div>

In [ ]:
rid = burst_train[burst_train.class10 == "imbalance"].sort_values("peakz_mic").iloc[-10].rec_id
raw_mic = load(rid)[:, MIC]
clean_mic, n_b = remove_mic_bursts(raw_mic, pre.mic.burst_z, pre.mic.burst_pad)
k = int(np.argmax(np.abs(raw_mic - np.median(raw_mic))))
t = np.arange(raw_mic.size) / FS
fig, axes = plt.subplots(1, 2, figsize=(15, 3.4))
axes[0].plot(t, raw_mic, lw=0.3, color="#bbb", label="raw")
axes[0].plot(t, clean_mic, lw=0.3, color="#8172b2", label="cleaned")
axes[0].set(xlabel="time (s)", ylabel="microphone (V)", title=f"{rid}: {n_b} bursts removed")
axes[0].legend(loc="upper right")
sl = slice(k - 150, k + 150)
axes[1].plot(t[sl] * 1e3, raw_mic[sl], "o-", ms=2, lw=0.8, color="#bbb", label="raw")
axes[1].plot(t[sl] * 1e3, clean_mic[sl], lw=1.2, color="#8172b2", label="cleaned")
axes[1].set(xlabel="time (ms)", ylabel="microphone (V)", title="largest burst, zoomed", ylim=(-1.5, 1.5))
fig.tight_layout()
fig.savefig(FIG / "02_mic_before_after.png")
print(f"peak robust z: {robust_z(raw_mic).max():.0f} → {robust_z(clean_mic).max():.1f}")

### Shortcut test
Can a model name the class from microphone peak statistics alone (max |x|, peak robust z, crest factor, kurtosis per
recording)? RandomForest, 5-fold CV by `cv_fold` on train + val (folds are by recording), macro-F1 over 10 classes
(chance ≈ 0.10). Before vs after burst removal, and the burst flag on its own.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Tests whether a model could guess the class from microphone spike statistics alone. It trains a random forest with 5-fold cross-validation, before and after burst removal, and reports the macro-F1 score (0.10 would be pure guessing). It also measures how reliably "this file has bursts" points to the two affected classes.</div>

In [ ]:
y = mic.class10.to_numpy()
feats = ["max_abs", "peakz", "crest", "kurtosis"]


def oof_predict(columns: list[str]) -> np.ndarray:
    X = mic[columns].to_numpy()
    pred = np.empty(len(mic), dtype=object)
    for f in range(5):
        tr, te = (mic.cv_fold != f).to_numpy(), (mic.cv_fold == f).to_numpy()
        pred[te] = RandomForestClassifier(300, random_state=0, n_jobs=-1).fit(X[tr], y[tr]).predict(X[te])
    return pred


rows = {}
for name, cols in {"raw mic (4 stats)": [f"raw_{f}" for f in feats],
                   "cleaned mic (4 stats)": [f"clean_{f}" for f in feats],
                   "raw, shape only (peakz, crest, kurtosis)": [f"raw_{f}" for f in feats[1:]],
                   "cleaned, shape only": [f"clean_{f}" for f in feats[1:]]}.items():
    p = oof_predict(cols)
    per = f1_score(y, p, average=None, labels=CLASS10)
    rows[name] = {"macro-F1": f1_score(y, p, average="macro"),
                  "F1 imbalance": per[CLASS10.index("imbalance")],
                  "F1 underhang_cage": per[CLASS10.index("underhang_cage")]}
shortcut = pd.DataFrame(rows).T.round(3)
has = mic.n_bursts > 0
in_pair = mic.class10.isin(["imbalance", "underhang_cage"])
print(f"burst present → imbalance or underhang_cage: precision {precision_score(in_pair, has):.3f}, "
      f"recall {has[in_pair].mean():.3f} ({has.sum()} bursty of {in_pair.sum()} such recordings)")
shortcut

## (b) The one > 1 % speed disagreement
`overhang__cage_fault__6g__16.1792` (train): tacho edges, tacho FFT and the accelerometer comb disagree by 1.3 %.
The instantaneous speed from each tacho interval shows why.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Looks at the one recording whose three speed estimates disagree. It plots the speed revolution by revolution, the tacho spectrum, and the region around the 4th shaft harmonic, with each method's estimate drawn as a dashed line.</div>

In [ ]:
rid = "overhang__cage_fault__6g__16.1792"
row = meta.set_index("rec_id").loc[rid]
x = load(rid)
ti = tacho_intervals(x[:, 0], pre)
inst = FS / (ti.intervals / ti.revs)
t_mid = (ti.edges[:-1] + ti.intervals / 2) / FS
est = {"tacho_edges": row.fr_tacho_edges, "tacho_fft": row.fr_tacho_fft, "accel_comb": row.fr_accel_comb}
colors = {"tacho_edges": "green", "tacho_fft": "orange", "accel_comb": "purple"}

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
axes[0].plot(t_mid, inst, "o-", ms=3, color="#333")
for k, v in est.items():
    axes[0].axhline(v, color=colors[k], ls="--", lw=1, label=f"{k} {v:.3f} Hz")
axes[0].set(xlabel="time (s)", ylabel="instantaneous shaft speed (Hz)", title="Speed per revolution")
axes[0].legend(fontsize=8)
f, a = amplitude_spectrum(x[:, 0], FS)
band = (f > 14.5) & (f < 17.5)
axes[1].plot(f[band], a[band], "o-", ms=3)
for k, v in est.items():
    axes[1].axvline(v, color=colors[k], ls="--", lw=1)
axes[1].set(xlabel="frequency (Hz)", ylabel="tacho amplitude (V)", title="Tacho spectrum (0.2 Hz bins)")
f, a = amplitude_spectrum(x[:, CH.index("over_radial")], FS)
band = (f > 3.8 * 15) & (f < 4.2 * 16.5)
axes[2].plot(f[band], a[band], lw=0.8)
for k, v in est.items():
    axes[2].axvline(4 * v, color=colors[k], ls="--", lw=1)
axes[2].set(xlabel="frequency (Hz)", ylabel="over_radial amplitude (V)", title="4th shaft harmonic region")
fig.tight_layout()
fig.savefig(FIG / "02_speed_disagreement.png")
print(f"instantaneous speed {inst.min():.2f}–{inst.max():.2f} Hz, mean {inst.mean():.3f}, std {inst.std():.3f} "
      f"({inst.std() / inst.mean():.1%}); clean intervals {ti.good.mean():.0%}")

**Verdict:** the speed wanders between about 15.5 and 16.4 Hz within the 5 s (±3 %), with every interval clean. The
tacho-edge estimate (15.754 Hz) is the true average over the recording, so the chosen speed is right. The FFT peak is
smeared over 15.4–16.2 Hz and the comb is pulled low by the smeared harmonics; neither is wrong, the recording is simply
non-stationary. Its order spectra will be blurred by about ±3 % per order. A per-recording speed-variability number
would find any others like it; that is a Stage 2 quality column, not done here.

## (c) Spectra

### Order spectra, one train recording per class
Frequency divided by measured shaft speed, so harmonics of the shaft sit on integers. One train recording per class,
nearest to 30 Hz, no quality flags. Imbalance should raise 1×; misalignment 2× (and axial energy); bearing faults barely
show here, because BPFO ≈ 3× and BPFI ≈ 5× coincide with shaft orders.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Plots vibration against shaft order (frequency divided by shaft speed) for one training recording per class and each of the six accelerometers. Red lines mark whole multiples of shaft speed, where imbalance and misalignment show up.</div>

In [ ]:
orders = np.arange(0, 10.5, 0.01)
fig, axes = plt.subplots(len(CLASS10), len(ACCELS), figsize=(18, 22), sharex=True)
for i, cls in enumerate(CLASS10):
    c = train[(train.class10 == cls) & train.quality_flags.isna()]
    r = c.loc[(c.fr_measured_hz - 30).abs().idxmin()]
    x = load(r.rec_id)
    for j, ch in enumerate(ACCELS):
        ax = axes[i, j]
        ax.semilogy(orders, order_spectrum(x[:, ch], FS, r.fr_measured_hz, orders), lw=0.6)
        for h in range(1, 11):
            ax.axvline(h, color="red", lw=0.4, alpha=0.4)
        if i == 0:
            ax.set_title(CH[ch])
        if j == 0:
            ax.set_ylabel(f"{cls}\n{r.fr_measured_hz:.1f} Hz\namp (V)", fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel("order (× shaft speed)")
fig.tight_layout()
fig.savefig(FIG / "02_order_spectra.png")

### Mean amplitude spectrum per channel
Geometric mean of the Welch amplitude spectral density over a stratified train subset (15 recordings per class, bursts
removed from the microphone). This is where the ~23 kHz "resonance" of CLAUDE.md §3 should appear.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Averages the frequency content of each channel over 150 training recordings and plots it, with a zoom on 21.5–22.5 kHz. The table lists each channel's biggest peak above 5 kHz, how far it rises above the background and how narrow it is.</div>

In [ ]:
ms = np.load(caches["meanspec"], allow_pickle=True)
f, asd = ms["freqs"], ms["asd"]
gmean = np.exp(np.log(asd + 1e-12).mean(axis=0))
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), gridspec_kw={"width_ratios": [3, 1]})
for i, ch in enumerate(CH):
    axes[0].loglog(f[1:], gmean[i, 1:], lw=0.7, label=ch)
    axes[1].plot(f, gmean[i] / np.median(gmean[i][(f > 5000) & (f < 20000)]), lw=0.8, label=ch)
axes[0].set(xlabel="frequency (Hz)", ylabel="amplitude spectral density (V/√Hz)", title=f"Mean ASD, n={asd.shape[0]}")
axes[0].legend(fontsize=7, ncol=2)
axes[1].set(xlim=(21500, 22500), yscale="log", xlabel="frequency (Hz)", ylabel="relative to 5–20 kHz median",
            title="zoom 21.5–22.5 kHz")
fig.tight_layout()
fig.savefig(FIG / "02_mean_asd.png")

hf = f > 5000
peak_bins = []
for i, ch in enumerate(CH):
    j = int(np.argmax(gmean[i][hf]))
    base = np.median(gmean[i][(f > 5000) & (f < 20000)])
    width = int((gmean[i][hf] > gmean[i][hf][j] / 2).sum())
    peak_bins.append({"channel": ch, "peak above 5 kHz (Hz)": f[hf][j], "× 5–20 kHz median": gmean[i][hf][j] / base,
                      "bins above half max": width})
print(f"bin width {f[1]:.1f} Hz")
pd.DataFrame(peak_bins).round(1)

### Which accelerometer hears which bearing?
Before choosing a demodulation band per channel: high-pass each accelerometer above 1 kHz (bearing impacts live there,
shaft harmonics do not) and compare its RMS and kurtosis for train 0 g bearing faults against train normal. If the
labels of CLAUDE.md §3 are right, underhang faults should light up the `under_*` channels.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Compares high-frequency vibration (above 1 kHz) for each bearing fault against normal recordings, on every accelerometer. Numbers above 1 mean that sensor picks up the fault; darker cells mean a stronger response.</div>

In [ ]:
sensor = pd.read_parquet(caches["sensor"])
acc = [CH[c] for c in ACCELS]
rms = sensor.groupby("class10")[[f"hp_rms_{c}" for c in acc]].median()
rms.columns = acc
ratio = (rms / rms.loc["normal"]).round(1)
order_rows = ["normal", "underhang_ball", "underhang_cage", "underhang_outer", "overhang_ball", "overhang_cage",
              "overhang_outer"]
ratio = ratio.loc[order_rows]
print("median > 1 kHz RMS relative to train normal (n per class:",
      sensor.class10.value_counts().reindex(order_rows).to_dict(), ")")
ratio.style.background_gradient(cmap="Blues", axis=None, vmin=0, vmax=10).format("{:.1f}")

### Spectral kurtosis → demodulation band
Spectral kurtosis measures how impulsive each frequency band is (0 for Gaussian noise). Bearing impacts excite a band;
the envelope should be taken there. Fixed grid: widths 1, 2, 4 and 8 kHz, stepped by half a width, lower edge ≥ 1 kHz.
Computed on train 0 g outer-race recordings (both bearing positions) and train normal recordings.

Two things decide the band (physics and train data only):
- A band must be **at least 2 kHz wide**: the envelope has to carry up to 12 orders at 61 Hz (≥ 1.46 kHz).
- Normal machines are impulsive in some bands too, so the score is the **contrast**: median SK of outer-race files minus
  median SK of normal files. Each channel is scored against the bearing position it hears best (see the swap finding
  below), and the band with the largest contrast wins.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> For each frequency band, measures how much more impulsive (spiky) outer-race recordings are than normal ones. It plots that contrast, then applies the project's band-selection rule (<code>ml.features.select_demod_band</code>) to the same training recordings and checks the result equals the band stored in the settings. It also checks that only <code>split_closed</code> train recordings were used.</div>

In [ ]:
sk = pd.read_parquet(caches["sk"])
sk["position"] = np.where(sk.class10 == "normal", "normal", sk.class10.str.split("_").str[0])
norm_sk = sk[sk.position == "normal"].groupby(["channel", "lo_hz", "hi_hz"]).sk.median()
contrast = (sk[sk.position != "normal"].groupby(["channel", "position", "lo_hz", "hi_hz"]).sk.median()
            - norm_sk).dropna().rename("contrast").reset_index()
contrast["width"] = contrast.hi_hz - contrast.lo_hz
contrast["centre"] = (contrast.hi_hz + contrast.lo_hz) / 2

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), sharey=True)
for ax, ch in zip(axes, ["under_radial", "over_radial"]):
    for pos, ls in [("underhang", "-"), ("overhang", "--")]:
        for w in sorted(contrast.width.unique()):
            c = contrast[(contrast.channel == ch) & (contrast.position == pos) & (contrast.width == w)]
            ax.plot(c.centre / 1e3, c.contrast, ls, marker=".", lw=0.9, label=f"{pos} outer, {w / 1e3:g} kHz")
    ax.axhline(0, color="k", lw=0.6)
    ax.set(title=ch, xlabel="band centre (kHz)")
axes[0].set_ylabel("median SK, outer race − normal")
axes[1].legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig(FIG / "02_sk_bands.png")

assert set(sk.rec_id) <= set(train.rec_id), "band selection may only see split_closed train recordings"
chosen = pd.DataFrame(select_demod_band(train, sk, pre)).T
for ch, r in chosen.iterrows():
    assert tuple(pre.envelope.bands_hz[ch]) == (r.lo_hz, r.hi_hz), f"configs/base.yaml band for {ch} is out of date"
chosen[["fault_class", "lo_hz", "hi_hz", "contrast", "n_fault", "n_reference"]]

### Selected bands against the sensor ratings
The accelerometers are rated only up to a point. MAFAULDA lists both models but does not say which one is on which
bearing, so every band is checked against both.

| Model | ±3 dB range | Resonance | Source |
| --- | --- | --- | --- |
| IMI 604B31 (triaxial) | 0.5–5,000 Hz | 10 kHz | spec sheet Revision B, ECN 49214 |
| IMI 601A01 | 0.27–10,000 Hz | 16 kHz | spec sheet Revision G, ECN 14824 |

Above the ±3 dB range the sensitivity is not flat and, near resonance, strongly amplified. That does not make a band
useless (impacts excite resonances), but a feature built there depends on how each sensor is mounted.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Reads the band chosen for every evaluation track from <code>results/demod_bands.json</code> and, for each band, says whether it lies inside, partly inside or outside the rated range of each sensor model, and whether it covers that model's resonance.</div>

In [ ]:
bands = json.loads((SYSTEM_CODE_DIR / "results" / "demod_bands.json").read_text())["tracks"]


def rating(lo: float, hi: float, spec) -> str:
    r_lo, r_hi = spec.range_3db_hz
    where = "inside" if r_lo <= lo and hi <= r_hi else ("outside" if lo >= r_hi else "partly outside")
    return where + (" (covers resonance)" if lo <= spec.resonance_hz <= hi else "")


rows = []
for track, t in bands.items():
    for ch, b in t["channels"].items():
        rows.append({"track": track, "channel": ch, "band (kHz)": f"{b['lo_hz'] / 1e3:g}–{b['hi_hz'] / 1e3:g}",
                     **{f"vs {m}": rating(b["lo_hz"], b["hi_hz"], s) for m, s in pre.envelope.sensors.items()}})
ratings = pd.DataFrame(rows)
ratings.drop_duplicates(subset=["channel", "band (kHz)"]).merge(
    ratings.groupby(["channel", "band (kHz)"]).track.apply(lambda s: ", ".join(s)).rename("tracks"),
    on=["channel", "band (kHz)"]).drop(columns="track")

Does the out-of-range band actually carry more bearing evidence than a band the sensors are rated for? On
**train** recordings only (0 g bearing faults and normal), compare each fault's own comb (outer race → BPFO, ball →
2×BSF, cage → FTF) in the selected closed-track band against the a priori in-range band 1–5 kHz. Values are the median
comb prominence minus the normal median (dB), with a one-sided Mann–Whitney p-value against normal. This is a report;
the bands are not switched.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads envelope spectra computed in both bands for the same train recordings, scores each fault's own comb in each band, and shows the difference from normal recordings side by side, as a table and a bar chart.</div>

In [ ]:
bc = np.load(caches["bandcompare"], allow_pickle=True)
bo, bspec, bcls, bchan, bband = bc["orders"], bc["spec"], bc["class10"], list(bc["channels"]), list(bc["bands"])
lines = comb_lines(pre)
own = {"outer": "bpfo", "ball": "bsf", "cage": "ftf"}
rows = []
for i in range(len(bcls)):
    for j, ch in enumerate(bchan):
        for k, band in enumerate(bband):
            rows.append({"class10": bcls[i], "channel": ch, "band": band,
                         **{f: line_prominence_db(bo, bspec[i, j, k], L, pre.comb.tol_orders,
                                                  pre.comb.bg_halfwidth_orders).mean() for f, L in lines.items()}})
bs = pd.DataFrame(rows)
res = []
for (ch, band), g in bs.groupby(["channel", "band"]):
    normal = g[g.class10 == "normal"]
    for cls in sorted(set(g.class10) - {"normal"}):
        fam = own[cls.split("_")[1]]
        v = g.loc[g.class10 == cls, fam]
        res.append({"channel": ch, "class10": cls, "comb": fam, "band": band,
                    "excess dB": v.median() - normal[fam].median(),
                    "p": mannwhitneyu(v, normal[fam], alternative="greater").pvalue})
band_cmp = pd.DataFrame(res).pivot_table(index=["channel", "class10", "comb"], columns="band",
                                         values=["excess dB", "p"])

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), sharey=True)
for ax, ch in zip(axes, bchan):
    t = band_cmp.loc[ch, "excess dB"]
    t.plot.bar(ax=ax, color=["#bbb", "#4c72b0"], width=0.8)
    ax.axhline(0, color="k", lw=0.6)
    ax.set(title=ch, xlabel="", ylabel="own comb − normal median (dB)")
    ax.set_xticklabels([f"{c}\n({f})" for c, f in t.index], rotation=0, fontsize=7)
fig.tight_layout()
fig.savefig(FIG / "02_band_comparison.png")
band_cmp.round(3)

### Envelope spectra
Band-pass in the chosen band → Hilbert envelope → spectrum, in orders. Median over train + val 0 g recordings of each
class. Cursors: FTF (cage), 2×BSF (ball), BPFO (outer race), BPFI (inner race); faint grey lines are integer shaft
orders. Each channel is shown with the bearing faults of the position it hears best.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Computes the envelope spectrum (the rhythm of the impacts inside the chosen band) for normal, outer-race, ball and cage recordings, and plots the typical one per class. Coloured dashed lines mark where each type of bearing fault should produce a peak.</div>

In [ ]:
env = np.load(caches["envelope"], allow_pickle=True)
eo, espec, ecls, echan = env["orders"], env["spec"], env["class10"], list(env["channels"])
cursors = {"FTF": (pre.bearing.ftf, "#2ca02c"), "2×BSF": (2 * pre.bearing.bsf, "#9467bd"),
           "BPFO": (pre.bearing.bpfo, "#d62728"), "BPFI": (pre.bearing.bpfi, "#ff7f0e")}
hears = {"under_radial": "overhang", "over_radial": "underhang"}
kinds = [("normal", "normal"), ("outer", "outer"), ("ball", "ball"), ("cage", "cage")]

fig, axes = plt.subplots(4, 2, figsize=(16, 12), sharex=True)
for j, ch in enumerate(echan):
    for i, (label, kind) in enumerate(kinds):
        cls = "normal" if kind == "normal" else f"{hears[ch]}_{kind}"
        s = np.median(espec[ecls == cls, j], axis=0)
        ax = axes[i, j]
        ax.semilogy(eo, s, lw=0.6, color="#333")
        for h in range(1, 13):
            ax.axvline(h, color="#ccc", lw=0.5)
        for name, (o, col) in cursors.items():
            for h in (1, 2, 3):
                ax.axvline(h * o, color=col, lw=0.9, ls="--", alpha=0.8, label=name if (i, j, h) == (0, 0, 1) else None)
        ax.set(xlim=(0, 12), title=f"{ch} · {cls} (n={int((ecls == cls).sum())})")
        if j == 0:
            ax.set_ylabel("envelope amp (V)")
axes[0, 0].legend(fontsize=8, ncol=4)
for ax in axes[-1]:
    ax.set_xlabel("order (× shaft speed)")
fig.tight_layout()
fig.savefig(FIG / "02_envelope_spectra.png")

## (d) `cage_fault` audit
Is `cage_fault` a cage fault, or an inner-race flaw as CLAUDE.md §3 suspects? For every train + val 0 g recording, score
four combs in the envelope-order spectrum: how far each comb's lines stand above their local background (dB, mean over
lines), minus the median of normal recordings on the same channel.

| Comb | Lines (orders) |
| --- | --- |
| FTF (cage) | FTF × 1–5 |
| BSF (ball) | 2×BSF, 2×BSF ± FTF |
| BPFO (outer race) | BPFO × 1–3 |
| BPFI (inner race) | BPFI × 1–2, each ± 1× shaft sideband |

**Caveat:** BPFO (2.998) and BPFI (5.002) sit within 0.006 orders of integer shaft orders, finer than the spectrum
resolves, so the BPFO and BPFI combs also measure shaft-harmonic modulation. Only FTF and 2×BSF are free of integers.
Outer race and ball are the positive controls, normal the negative control. This is a finding only: no label or
parameter changes because of it.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Scores every recording on four fault patterns (cage, ball, outer race, inner race): how far the expected peaks stand above their surroundings, minus the typical normal recording. The first table is the median score; the second is the share of recordings scoring higher than 95 % of normal ones.</div>

In [ ]:
lines = comb_lines(pre)
rows = []
for i in range(len(ecls)):
    if not (ecls[i] == "normal" or ecls[i].split("_")[0] in ("underhang", "overhang")):
        continue
    for j, ch in enumerate(echan):
        r = {"class10": ecls[i], "channel": ch}
        for fam, L in lines.items():
            r[fam] = line_prominence_db(eo, espec[i, j], L, pre.comb.tol_orders, pre.comb.bg_halfwidth_orders).mean()
        rows.append(r)
combs = pd.DataFrame(rows)
fams = list(lines)
ref = combs[combs.class10 == "normal"].groupby("channel")[fams]
excess = combs.copy()
excess[fams] = combs[fams] - ref.median().loc[combs.channel].to_numpy()
p95 = (combs[combs.class10 == "normal"].groupby("channel")[fams].quantile(0.95)
       - ref.median()).loc[combs.channel].to_numpy()
above = excess[fams].to_numpy() > p95

summary = excess.groupby(["channel", "class10"])[fams].median().round(2)
share = pd.DataFrame(above, columns=fams).assign(channel=excess.channel.values, class10=excess.class10.values)
share = share.groupby(["channel", "class10"])[fams].mean().round(2)
display(summary)
share

Median excess over normal (dB) per comb, on the channel that hears each bearing position, with Mann–Whitney p-values against normal:

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Draws those scores as box plots for the channel that hears each bearing position, and tests whether each fault class scores higher than normal (small p-values mean it clearly does).</div>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)
tests = []
for ax, ch in zip(axes, echan):
    classes = [f"{hears[ch]}_{k}" for k in ("outer", "ball", "cage")] + ["normal"]
    data = excess[excess.channel == ch]
    width = 0.2
    for k, fam in enumerate(fams):
        vals = [data.loc[data.class10 == c, fam] for c in classes]
        ax.boxplot(vals, positions=np.arange(len(classes)) + (k - 1.5) * width, widths=width * 0.9,
                   patch_artist=True, boxprops={"facecolor": f"C{k}", "alpha": 0.6}, showfliers=False,
                   medianprops={"color": "k"})
        for c, v in zip(classes[:-1], vals[:-1]):
            p = mannwhitneyu(v, data.loc[data.class10 == "normal", fam], alternative="greater").pvalue
            tests.append({"channel": ch, "class10": c, "comb": fam, "median excess dB": v.median(), "p (> normal)": p})
    ax.set_xticks(range(len(classes)), classes, rotation=15)
    ax.axhline(0, color="k", lw=0.6)
    ax.set(title=ch, ylabel="comb prominence − normal median (dB)")
axes[1].legend([plt.Rectangle((0, 0), 1, 1, fc=f"C{k}", alpha=0.6) for k in range(4)], fams, fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "02_cage_audit.png")
pd.DataFrame(tests).pivot_table(index=["channel", "class10"], columns="comb",
                                values=["median excess dB", "p (> normal)"]).round(3)

## Findings
- **Mic bursts are electrical and are removed.** Threshold z = 18.2 (1.25 × the train-normal maximum of 14.57) sits in a clean gap: the largest burst-free file is 14.57, the smallest caught file 21.18. It catches 168 files (143 train+val): imbalance 20–35 g (83), underhang cage 6–35 g (84) and one overhang cage 35 g (val). Bursts are 1–2 samples up to about −160 V with a ~0.2 ms anti-alias-filter ring, median run 7.5 samples, and no accelerometer response (median robust z 2.5 at a burst, 2.5 at random times).
- **The burst shortcut is real but small; mic peak statistics stay informative.** A burst means imbalance or underhang cage with 99.3 % precision (32 % recall). Removing bursts lowers the RandomForest macro-F1 on mic peak statistics only from 0.628 to 0.620 (imbalance F1 0.65 → 0.63, underhang cage 0.76 → 0.73); crest, kurtosis and peak z alone still reach 0.48, far above chance (0.10). Rule 10's ban on these features is needed for that reason, not only because of the bursts.
- **The ~23 kHz "resonance" is a narrowband electrical line**: 22,028 Hz, 2–3 bins (6.1 Hz each) wide, with sidebands at ±55–61 Hz and ±116–122 Hz (60 Hz mains and its harmonic at this resolution), on every accelerometer (7.3–61× the 5–20 kHz median) and the microphone (168×). It is not a structural resonance, and both demodulation bands avoid it.
- **The accelerometer groups may be swapped relative to CLAUDE.md §3.** Above 1 kHz, underhang ball raises the *overhang* axial and tangential channels 11× and underhang outer raises *overhang* radial 5×, while overhang ball raises *underhang* tangential 42×. Underhang tangential responds to most faults, so this is strong evidence, not proof. Demodulation bands follow the data: **under_radial 9–11 kHz** (hears overhang faults, SK contrast +0.94) and **over_radial 13–17 kHz** (hears underhang faults, +4.31).
- **`cage_fault` shows no inner-race signature.** On the channel that hears each position, the BPFI comb (integer orders with ±1× sidebands) is 1.3 dB (underhang, p = 0.80) and 2.4 dB (overhang, p = 1.0) *below* normal, whereas a real race defect (underhang outer) lifts the same comb by 10.2 dB (p < 0.001). The cage signature is weak too: FTF +1.1 dB for underhang cage (p = 0.37), −0.4 dB for overhang cage, which only shows a small 2×BSF excess (+0.95 dB, p = 0.010). Controls are mixed: underhang outer (BPFO +9.0 dB) and overhang ball (2×BSF +2.5 dB, FTF +3.4 dB) pass, underhang ball fails. Labels stay as they are.
- **Envelope combs cannot separate outer from inner race here**: BPFO and BPFI fall within 0.006 orders of integer shaft orders, and underhang outer lifts every integer order from 1 to 12. The one > 1 % speed-disagreement file is a non-stationary recording (15.47–16.35 Hz within 5 s, std 1.5 %); its chosen speed, the tacho-edge average 15.754 Hz, is right.


## Outputs written
- Figures: `results/figures/02_mic_peakz_hist.png`, `02_mic_burst_zoom.png`, `02_mic_before_after.png`,
  `02_speed_disagreement.png`, `02_order_spectra.png`, `02_mean_asd.png`, `02_sk_bands.png`,
  `02_envelope_spectra.png`, `02_cage_audit.png`
- `results/mic_burst_files.csv`: every file the burst threshold catches
- Parameters (in `configs/base.yaml` and `ml/config.py`, asserted above): `mic.burst_z`, `envelope.bands_hz`;
  comb definitions in `comb`